# Day 6 Exercises: Regression and Neural Networks

Student version with gaps.

## Context

This exercise session starts from the mass-spring-damper model
$$
    y''(t) = -\frac{k}{m} y(t) - \frac{c}{m} y'(t),
    \qquad y(0)=y_0,\quad y'(0)=v_0.
$$

The exact solution is only used as a data generator. All examples are regression examples, so every model is trained and evaluated through the same shared pipeline.

> #### Instructions
> 
> Run the setup cells first. In the exercises, change only the requested data, model initialization, model application, or loss functions. Use the shared tables and plots to answer the open questions.


## Learning Goals

The lab is designed to separate modelling choices from training and plotting boilerplate.

## Learning goals

By the end of the lab you should be able to:

- compare underfitting and overfitting by varying noise level and polynomial degree;
- reduce polynomial overfitting by changing the loss to ridge regression;
- construct piecewise linear models with fixed ReLU grid nodes;
- make the ReLU grid nodes trainable;
- make the model deeper by adding more than one hidden layer;
- compare all models with the same training, plotting, and diagnostic code.



## Shared Regression Workflow

### Notebook convention

Blocks labelled **Instructions** tell you what to do. Blocks labelled **Open questions** are the questions to answer after running the diagnostics.

### Structure
This notebook is organized around one reusable regression workflow. The setup cell defines the common tools once; the exercise sections change only the modelling ingredients being compared.

### What stays fixed?
The experiment format, Adam trainer, prediction wrapper, train/test error table, and plotting helpers are shared by all models. This keeps the comparisons fair: when diagnostics change, the intended cause should be the data, model, initialization, or loss.

### What changes between exercises?
Each exercise chooses a dataset, an `init_model(key)` function, an `apply_model(params, x)` function, and a loss. Provided infrastructure is imported from `day6_helpers.py`; the code students edit appears directly in the relevant exercise sections.

> #### Instructions
> Every exercise uses the same pattern:
>
> 1. choose a regression dataset, such as `clean_data`, `medium_noise_data`, or `high_noise_data`;
> 2. choose `init_model(key)`;
> 3. choose `apply_model(params, x)`;
> 4. choose `loss(params, apply_model, x, y)`;
> 5. call `make_experiment(...)`, `run_experiments(...)`, and the shared plotting helpers;
> 6. inspect the error table and fit plots before answering the open questions.

### Optional diagnostic
Adam is provided here so the lab can focus on regression, noise, model flexibility, and regularization. Loss-history plots are available through `plot_training_histories`, but they are disabled by default. Set `SHOW_OPTIMIZATION_PLOTS = True` in the setup cell if you want to inspect them.


## Setup

The shared data, optimizer, evaluation, and plotting helpers live next to this notebook in `day6_helpers.py`. The notebook imports those provided pieces here; the model definitions that students edit appear inside the exercise sections below.

You may open `day6_helpers.py` if you want to see how the provided data, training loop, tables, and plots are implemented. This is optional: the core lab does not require reading or editing the helper file.

> #### Instructions
>
> Run this cell before starting the exercises. You do not need to edit the helper module during the core lab.

In [ ]:
from pathlib import Path
import sys

import numpy as np
import jax.numpy as jnp
from jax import random

# Make the local helper module importable when running from the day folder,
# the notebooks folder, the quarto folder, or the course root.
for helper_dir in [
    Path.cwd(),
    Path.cwd() / "day6",
    Path.cwd() / "notebooks" / "day6",
    Path.cwd() / "quarto" / "notebooks" / "day6",
]:
    if (helper_dir / "day6_helpers.py").exists():
        sys.path.insert(0, str(helper_dir))
        break

from day6_helpers import (
    clean_data,
    medium_noise_data,
    high_noise_data,
    mean_squared_error_loss,
    make_experiment,
    run_experiments,
    print_model_table,
    plot_datasets,
    plot_model_fits,
    plot_training_histories,
)

## Data Inspection

The next code cell visualizes the clean, medium-noise, and high-noise versions of the same mass-spring-damper data set.

> #### Instructions
>
> Run the cell and compare the three panels before moving to Exercise 1.

In [ ]:
# First look at the data. All panels come from the same clean oscillator; only
# the observation noise changes between them.
plot_datasets(
    [clean_data, medium_noise_data, high_noise_data],
    ["clean", "medium noise", "high noise"],
    title="Regression datasets used throughout the notebook",
)

## Exercise 1: Polynomial Regression, Noise, and Ridge

Polynomial regression represents the fitted curve as
$$
f_\theta(x) = \sum_{j=0}^p \theta_j x^j.
$$

In this exercise, the polynomial degree `p`, the noise level, and the ridge parameter are the modelling choices under comparison.

> #### Instructions
>
> 1. Complete the clearly marked `STUDENT TASK` / `YOUR CODE HERE` blocks in the next code cell.
> 2. Run the code cell to check `polynomial_features(...)` and `ridge_polynomial_loss(...)`.
> 3. Then run the comparison and ridge diagnostics below.
> 4. Use the shared error tables and fit plots for your answers.

> #### Open questions
> 1. For which combinations do the training and test errors tell noticeably different stories?
> 2. Where does increasing the polynomial degree improve the fit, and where does it change the fit in a less useful way?
> 3. In the ridge diagnostic below, change the entries of `ridge_values`. How does the ridge parameter affect the degree-32 fit on the high-noise data?

In [ ]:
def polynomial_features(x, degree):
    """Return the polynomial feature matrix [1, x, ..., x**degree]."""
    x = jnp.asarray(x)
    # Each row is one sample; each column is one basis function.
    # ======================================================================
    # STUDENT TASK: polynomial feature matrix
    # Replace the placeholder line below with code that returns
    # columns 1, x, x**2, ..., x**degree with shape (len(x), degree + 1).
    # Hint: Think of each column as one polynomial degree.
    # YOUR CODE HERE
    # ======================================================================
    raise NotImplementedError("Implement the polynomial feature matrix before running Exercise 1.")


def init_polynomial_model(key, degree):
    """Initialize polynomial coefficients."""
    return {"weights": jnp.zeros(degree + 1)}


def apply_polynomial_model(params, x, degree):
    """Evaluate the polynomial model on a column input array."""
    features = polynomial_features(x[:, 0], degree)
    return features @ params["weights"]


def ridge_polynomial_loss(params, apply_model, x, y, regularization):
    """Mean-square regression loss with an L2 penalty on non-constant weights."""
    # The intercept is not penalized. Penalizing it would shift the whole curve
    # toward zero, which is not the behaviour we want to study here.
    # ======================================================================
    # STUDENT TASK: ridge-regularized polynomial loss
    # Replace the placeholder line below with code that returns
    # MSE + regularization * sum(weights[1:]**2).
    # Hint: Keep the provided data-fit loss and the coefficient penalty conceptually separate.
    # YOUR CODE HERE
    # ======================================================================
    raise NotImplementedError("Implement the ridge-regularized polynomial loss for Exercise 1b.")


# Quick checks for the Exercise 1 functions. These catch the most common
# mistakes: wrong feature shape, wrong column order, or a ridge loss that does
# not reduce to MSE when the regularization parameter is zero.
A_check = polynomial_features(jnp.array([-1.0, 0.0, 1.0]), degree=3)
assert A_check.shape == (3, 4)
assert np.allclose(np.asarray(A_check[1]), np.array([1.0, 0.0, 0.0, 0.0]))
assert np.allclose(np.asarray(A_check[2]), np.array([1.0, 1.0, 1.0, 1.0]))
params_check = init_polynomial_model(random.PRNGKey(1), degree=4)
apply_check = lambda params, x: apply_polynomial_model(params, x, degree=4)
x_check = jnp.asarray(high_noise_data["x_train"][:, None])
y_check = jnp.asarray(high_noise_data["y_train"])
assert np.allclose(
    float(ridge_polynomial_loss(params_check, apply_check, x_check, y_check, regularization=0.0)),
    float(mean_squared_error_loss(params_check, apply_check, x_check, y_check)),
)

> #### Instructions
>
> Run the polynomial comparison. This cell keeps the data, optimizer family, and plotting path fixed while varying only the noise level and polynomial degree.

In [ ]:
# 1a: try every combination of noise level and polynomial degree.
noise_cases = [
    ("clean", clean_data),
    ("medium noise", medium_noise_data),
    ("high noise", high_noise_data),
]
polynomial_degrees = [2, 8, 32]

# Degree-32 models get extra steps and a smaller learning rate so their behavior
# is about flexibility/noise rather than incomplete or unstable optimization.
polynomial_steps = {2: 1200, 8: 1600, 32: 6000}
polynomial_learning_rates = {2: 2e-2, 8: 2e-2, 32: 5e-3}

polynomial_cases = [
    (f"{noise_label}, degree {degree}", data, noise_label, degree)
    for noise_label, data in noise_cases
    for degree in polynomial_degrees
]

poly_experiments = []
for seed, (name, data, noise_label, degree) in enumerate(polynomial_cases, start=10):
    # Capture degree as a default argument. Otherwise every lambda would use
    # the final loop value, which is a classic small Python trap.
    poly_experiments.append(make_experiment(
        name=name,
        data=data,
        init_model=lambda key, degree=degree: init_polynomial_model(key, degree),
        apply_model=lambda params, x, degree=degree: apply_polynomial_model(params, x, degree),
        loss=mean_squared_error_loss,
        seed=seed,
        steps=polynomial_steps[degree],
        learning_rate=polynomial_learning_rates[degree],
    ))

poly_models = run_experiments(poly_experiments)
print_model_table(poly_models)
plot_training_histories(
    poly_models,
    title="Polynomial models: all noise levels and degrees",
)

poly_models_by_noise = {}
for (_, _, noise_label, _), model in zip(polynomial_cases, poly_models):
    poly_models_by_noise.setdefault(noise_label, []).append(model)

for noise_label, data in noise_cases:
    plot_model_fits(
        poly_models_by_noise[noise_label],
        data,
        title=f"{noise_label}: polynomial degrees 2, 8, 32",
    )

> #### Instructions
>
> Now test the regularized variant. This cell keeps the high-degree noisy polynomial setting fixed and changes only the ridge penalty.

In [ ]:
# 1b: keep the high-degree noisy model, change only the ridge parameter.
# Change this list to test other ridge parameters.
ridge_values = [0.0, 3e-5, 1e-4, 1e-3]

ridge_experiments = []
for seed, ridge_lambda in enumerate(ridge_values, start=30):
    # make_experiment has no separate ridge field. The loss function below
    # passes the current ridge_lambda to ridge_polynomial_loss(...).
    ridge_experiments.append(make_experiment(
        name=f"degree 32, ridge lambda={ridge_lambda:g}",
        data=high_noise_data,
        init_model=lambda key: init_polynomial_model(key, degree=32),
        apply_model=lambda params, x: apply_polynomial_model(params, x, degree=32),
        loss=lambda params, apply_model, x, y, ridge_lambda=ridge_lambda: ridge_polynomial_loss(
            params, apply_model, x, y, regularization=ridge_lambda
        ),
        seed=seed,
        steps=polynomial_steps[32],
        learning_rate=polynomial_learning_rates[32],
    ))

ridge_models = run_experiments(ridge_experiments)
print_model_table(ridge_models, high_noise_data)
plot_training_histories(ridge_models, title="Ridge regression: same model, different loss")
plot_model_fits(ridge_models, high_noise_data, title="Ridge regularization and overfitting")

## Exercise 2: Piecewise Linear ReLU Models

A fixed-grid ReLU model is a piecewise linear model of the form
$$
    g(x)=a_0+a_1x+\sum_{j=1}^M b_j\,\mathrm{ReLU}(x-\tau_j), \qquad \mathrm{ReLU}(z)=\max(0,z).
$$

With fixed grid nodes, only the output weights are trained. With trainable grid nodes, the hinge locations `tau_j` also move during training.

> #### Instructions
> 1. Complete the clearly marked `STUDENT TASK` / `YOUR CODE HERE` blocks in the next code cell.
> 2. Run the cell to check the ReLU and feature-map implementations and to configure the two experiments.
> 3. Then run the diagnostic cell below and inspect the fit plot plus the printed initial/trained knot locations.

> #### Open questions
> 1. Where do the fixed-grid and trainable-grid curves differ most visibly?
> 2. How do the trained knot locations compare with the initial equidistant grid?
> 3. Does making the knots trainable mainly change the training error, the test error, or the shape of the fitted curve?

In [ ]:
def relu_jax(z):
    """JAX implementation of ReLU(z) = max(0, z)."""
    # ======================================================================
    # STUDENT TASK: JAX ReLU
    # Replace the placeholder line below with a JAX expression for
    # max(z, 0).
    # Hint: Use an elementwise nonlinearity that works on JAX arrays.
    # YOUR CODE HERE
    # ======================================================================
    raise NotImplementedError("Implement the JAX ReLU before running Exercise 2.")


def fixed_relu_features(x, knots):
    """Feature matrix for a one-dimensional piecewise linear ReLU model."""
    x = jnp.asarray(x)
    knots = jnp.asarray(knots)
    # ======================================================================
    # STUDENT TASK: fixed-grid ReLU feature matrix
    # Replace the placeholder line below with code that returns columns
    # 1, x, ReLU(x-knot_1), ..., ReLU(x-knot_M).
    # Hint: Build one column per hinge function, together with the intercept and linear terms.
    # YOUR CODE HERE
    # ======================================================================
    raise NotImplementedError("Implement the fixed-grid ReLU feature matrix for Exercise 2a.")


def init_fixed_relu_grid_model(key, knots):
    """Initialize a shallow ReLU model with fixed grid nodes."""
    return {"weights": jnp.zeros(2 + len(knots))}


def apply_fixed_relu_grid_model(params, x, knots):
    """Evaluate the fixed-grid ReLU model."""
    features = fixed_relu_features(x[:, 0], knots)
    return features @ params["weights"]


def init_trainable_relu_grid_model(key, knots):
    """Initialize a shallow ReLU model whose grid nodes can move."""
    key_weights, _ = random.split(key)
    return {
        "weights": 0.05*random.normal(key_weights, shape=(2 + len(knots),)),
        "knots": jnp.asarray(knots),
    }


def apply_trainable_relu_grid_model(params, x):
    """Evaluate the ReLU grid model using the current trainable knot locations."""
    # ======================================================================
    # STUDENT TASK: trainable-grid ReLU model
    # Replace the placeholder line below with code that uses
    # params["knots"] inside fixed_relu_features and applies params["weights"].
    # Hint: Treat the current knot locations as part of the model state.
    # YOUR CODE HERE
    # ======================================================================
    raise NotImplementedError("Implement the trainable-grid ReLU model for Exercise 2b.")


# Quick checks for the Exercise 2 functions. The feature check only uses one
# knot so that the expected matrix shape is easy to reason about.
assert np.allclose(np.asarray(relu_jax(jnp.array([-2.0, 0.0, 3.0]))), np.array([0.0, 0.0, 3.0]))
B_check = fixed_relu_features(jnp.array([-1.0, 0.0, 1.0]), knots=jnp.array([0.0]))
assert B_check.shape == (3, 3)

knots = jnp.linspace(-0.9, 0.9, 14)

# The two experiments below share the data, loss, optimizer, and plotting code.
# The only difference is whether the knot locations are fixed or trainable.
relu_grid_experiments = [
    make_experiment(
        name="fixed equidistant ReLU grid",
        data=medium_noise_data,
        init_model=lambda key: init_fixed_relu_grid_model(key, knots),
        apply_model=lambda params, x: apply_fixed_relu_grid_model(params, x, knots),
        loss=mean_squared_error_loss,
        seed=40,
        steps=1200,
        learning_rate=2e-2,
    ),
    make_experiment(
        name="trainable ReLU grid nodes",
        data=medium_noise_data,
        init_model=lambda key: init_trainable_relu_grid_model(key, knots),
        apply_model=apply_trainable_relu_grid_model,
        loss=mean_squared_error_loss,
        seed=41,
        steps=1200,
        learning_rate=2e-2,
    ),
]

> #### Instructions
>
> Run the shared diagnostics for the fixed-grid and trainable-grid ReLU models. Use the table, fitted curves, and printed knots for the open questions.

In [ ]:
relu_grid_models = run_experiments(relu_grid_experiments)
fixed_relu_model, trainable_relu_grid_model = relu_grid_models

print_model_table(relu_grid_models, medium_noise_data)
plot_training_histories(relu_grid_models, title="Fixed vs trainable ReLU grid nodes")
plot_model_fits(
    relu_grid_models,
    medium_noise_data,
    title="Piecewise linear ReLU models on medium-noise data",
)

print("Initial equidistant knots:")
print(np.asarray(knots))
print("Trained knots:")
print(np.asarray(trainable_relu_grid_model["params"]["knots"]))

## Exercise 3: Deeper ReLU Model

The shallow trainable-grid model in Exercise 2 uses one ReLU feature layer. Here we use two hidden ReLU layers before the final affine output layer, while keeping the same data, loss, optimizer, tables, and plotting helpers.

> #### Instructions
> 1. Complete the clearly marked `STUDENT TASK` / `YOUR CODE HERE` block in the next code cell.
> 2. Run the cell to check that the network output has the expected shape and to train the model.
> 3. Compare the deeper model with the fixed-grid and trainable-grid shallow ReLU models.
> 4. Base your answers on the shared error table and fit plot.

> #### Open questions
>
> 1. How do the train and test errors of the deeper model compare with the shallow ReLU models?
> 2. In which time intervals does the deeper model change the fitted curve most noticeably?
> 3. Based on the diagnostics, is the extra flexibility useful for this medium-noise data set?

In [ ]:
def init_deep_relu_network(key, width=10):
    """Initialize a small two-hidden-layer ReLU network."""
    key_w1, key_w2, key_w3 = random.split(key, 3)
    return {
        "W1": 1.5*random.normal(key_w1, shape=(1, width)),
        "b1": jnp.linspace(-1.0, 1.0, width),
        "W2": random.normal(key_w2, shape=(width, width))/jnp.sqrt(width),
        "b2": jnp.zeros(width),
        "W3": 0.15*random.normal(key_w3, shape=(width,)),
        "b3": jnp.array(0.0),
    }


def apply_deep_relu_network(params, x):
    """Evaluate an affine-ReLU-affine-ReLU-affine network."""
    # x has shape (n_samples, 1). Return shape (n_samples,).
    # ======================================================================
    # STUDENT TASK: deeper ReLU forward pass
    # Replace the placeholder line below with the forward pass
    # affine -> ReLU -> affine -> ReLU -> affine.
    # Hint: Apply the affine maps and activations in the order described above.
    # YOUR CODE HERE
    # ======================================================================
    raise NotImplementedError("Implement the deeper ReLU network for Exercise 3.")


# Shape check before running the optimizer. A model that returns shape (n, 1)
# instead of (n,) would silently create awkward broadcasting in the loss.
x_shape_check = jnp.asarray(medium_noise_data["x_train"][:, None])
deep_check = apply_deep_relu_network(init_deep_relu_network(random.PRNGKey(50)), x_shape_check)
assert deep_check.shape == jnp.asarray(medium_noise_data["y_train"]).shape

# The deeper network uses the same regression pipeline as the previous models.
# We lower the learning rate because the parameterization is more nonlinear.
deep_experiment = make_experiment(
    name="two-hidden-layer ReLU network",
    data=medium_noise_data,
    init_model=lambda key: init_deep_relu_network(key, width=10),
    apply_model=apply_deep_relu_network,
    loss=mean_squared_error_loss,
    seed=50,
    steps=1400,
    learning_rate=2e-3,
)
deep_relu_model = run_experiments([deep_experiment])[0]

print_model_table([fixed_relu_model, trainable_relu_grid_model, deep_relu_model], medium_noise_data)
plot_training_histories(
    [trainable_relu_grid_model, deep_relu_model],
    title="Trainable grid vs deeper ReLU network",
)
plot_model_fits(
    [fixed_relu_model, trainable_relu_grid_model, deep_relu_model],
    medium_noise_data,
    title="Piecewise linear, trainable grid, and deeper ReLU models",
)

## Reflection and Deliverable

> #### Instructions
>
> Before you finish, make sure that all implementation gaps have been completed and all assertions pass. Then write short answers to the prompts below.

> #### Reflection questions
>
> 1. In Exercise 1a, which polynomial cases show underfitting and which show overfitting?
> 2. In Exercise 1b, how does ridge regularization change the high-degree noisy polynomial fit?
> 3. In Exercise 2, what changes when the ReLU grid nodes become trainable?
> 4. In Exercise 3, how does the deeper model compare with the fixed-grid and trainable-grid ReLU models?
> 5. Which pieces changed between exercises, and which training/evaluation code stayed fixed?

> #### Optional extensions
> - change the medium or high noise level;
> - change the number of ReLU grid nodes;
> - change the width of the deeper network;
> - add one more hidden layer and compare with Exercise 3.